In [0]:
from pyspark.sql import functions as F 

In [0]:
trip_df = spark.table("samples.nyctaxi.trips")


trip_df.display()

In [0]:
# # with column functions

trip_df = trip_df.withColumn('fare_amount_Rupee',F.col('fare_amount')*92)
trip_df.display()


In [0]:
trip_df = trip_df.withColumnRenamed("fare_amount_Rupee","fare_amount_INR")

trip_df.display()

In [0]:
trip_df = trip_df. \
withColumn("Dummy1",F.lit("0")) \
        .withColumn("Dummy2",F.lit("1"))

trip_df.display()

In [0]:
%sql
select trip_distance, fare_amount, tpep_pickup_datetime, tpep_dropoff_datetime  

from samples.nyctaxi.trips;

In [0]:
trip_df = spark.sql("SELECT * from samples.nyctaxi.trips")

trip_df.display()

In [0]:
trip_df = spark.sql("select fare_amount * 92 as fare_amount_INR from samples.nyctaxi.trips")

trip_df.display()

In [0]:

trip_df1 = spark.table('samples.nyctaxi.trips')
display(trip_df1)




In [0]:
df1 = trip_df1.select(F.col("trip_distance"), F.col("fare_amount").alias("fare_USD"))

display(df1)

In [0]:
df2 = trip_df1.select("trip_distance", "fare_amount", "pickup_zip")
display(df2)

In [0]:
df2 = trip_df1.filter(F.col("fare_amount") > 15)

df2.display()

In [0]:
#

In [0]:
df2 = df2.withColumnRenamed("fare_amount", "fare_amount_usd")
display(df2)

In [0]:
df2 = df2.withColumn("fare_amount_inr", F.col("fare_amount_usd") * 92)

display(df2)

In [0]:
df2 = df2.withColumn("Dummy1",F.lit("1"))
df2 = df2.withColumn("Dummy",F.lit("1"))
df2 = df2.withColumn("Dummy2",F.lit("1"))

df2.display()

In [0]:
df2 = df2.drop("Dummy")

df2.display()


# GroupBy & AGG() Function
## APPLE SALES Dataset 

In [0]:
apple_df = spark.table('datasets.apple_sales_delta')

apple_df.display()

In [0]:
apple_df = (
    apple_df
    .groupBy("country", "year")
    .agg(F.sum("revenue_usd").alias("total_revenue_USD"))
    .orderBy(F.col("year"))
)

display(apple_df)

In [0]:
apple_df.createOrReplaceTempView("apple_sales_temp")
display(spark.table("apple_sales_temp"))

In [0]:
# You can perform various operations on the temp table "apple_sales_temp" such as:
# 1. Select specific columns
display(spark.sql("SELECT country, year, total_revenue_USD FROM apple_sales_temp"))

# 2. Filter rows
display(spark.sql("SELECT * FROM apple_sales_temp WHERE total_revenue_USD > 1000000"))

# 3. Aggregate data
display(spark.sql("SELECT country, SUM(total_revenue_USD) AS sum_revenue FROM apple_sales_temp GROUP BY country"))

# 4. Order results
display(spark.sql("SELECT * FROM apple_sales_temp ORDER BY total_revenue_USD DESC"))

# 5. Join with other tables
display(spark.sql("""
SELECT a.*, b.other_column
FROM apple_sales_temp a
JOIN other_table b ON a.country = b.country
"""))

# 6. Add calculated columns
display(spark.sql("SELECT *, total_revenue_USD * 82 AS total_revenue_INR FROM apple_sales_temp"))

In [0]:
# Previous code: Aggregates revenue_usd directly from the base table.
display(spark.sql("SELECT country, year, sum(revenue_usd) AS total_revenue_USD FROM datasets.apple_sales_delta GROUP BY country, year"))

# Fixed code: Would aggregate from the temp view 'apple_sales_temp', which already contains grouped and summed data.
# Example:
# display(spark.sql("SELECT country, year, total_revenue_USD FROM apple_sales_temp"))

In [0]:
# Repartition: Increases or decreases the number of partitions in the DataFrame.
apple_df_repartitioned = apple_df.repartition(10)

# Coalesce: Reduces the number of partitions, typically used after filtering or aggregation.
apple_df_coalesced = apple_df.coalesce(2)

display(apple_df_repartitioned)
display(apple_df_coalesced)

In [0]:
from pyspark.sql import Window
import pyspark.sql.functions as F

# Aggregate revenue by country and year
apple_df_agg = (
    apple_df
    .groupBy("country", "year")
    .agg(F.sum("revenue_usd").alias("total_revenue_USD"))
)

# Window specification: partition by country, order by year
window_spec = Window.partitionBy("country").orderBy("year")

# Apply window functions
apple_df_window = (
    apple_df_agg
    # Ranking functions
    .withColumn("rank", F.rank().over(window_spec))           # Standard rank
    .withColumn("dense_rank", F.dense_rank().over(window_spec)) # Dense rank
    .withColumn("row_number", F.row_number().over(window_spec)) # Row number

    # Cumulative sum of revenue for each country up to current year
    .withColumn(
        "cumulative_revenue",
        F.sum("total_revenue_USD").over(
            window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )

    # Minimum revenue up to current year for each country
    .withColumn(
        "min_revenue",
        F.min("total_revenue_USD").over(
            window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )

    # Maximum revenue up to current year for each country
    .withColumn(
        "max_revenue",
        F.max("total_revenue_USD").over(
            window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )

    # Average revenue up to current year for each country
    .withColumn(
        "avg_revenue",
        F.avg("total_revenue_USD").over(
            window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )

    # Previous year's revenue for each country
    .withColumn("lag_revenue", F.lag("total_revenue_USD", 1).over(window_spec))

    # Next year's revenue for each country
    .withColumn("lead_revenue", F.lead("total_revenue_USD", 1).over(window_spec))
)

display(apple_df_window)

In [0]:
# Apply window functions
apple_df_window = (
    apple_df_agg
    # Ranking functions
    .withColumn("rank", F.rank().over(window_spec))           # Standard rank
    .withColumn("dense_rank", F.dense_rank().over(window_spec)) # Dense rank
    .withColumn("row_number", F.row_number().over(window_spec)) # Row number

    # Cumulative sum of revenue for each country up to current year
    .withColumn(
        "cumulative_revenue",
        F.sum("total_revenue_USD").over(
            window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )
)

apple_df_window.display()